In [1]:
import pandas as pd
import os
import sys


module_directory = '.' # Assuming financial_analysis.py is in the current directory or already accessible
if 'financial_analysis' not in sys.modules:
    
    if os.path.exists('../financial_analysis.py'):
        sys.path.insert(0, '..')
    else:
        sys.path.insert(0, '.')
    print("System path updated for module import.")


# --- STEP 1: IMPORT THE REFACTORED CLASS ---
# Import the FinancialAnalyzer class from the updated module
from financial_analysis import FinancialAnalyzer

# --- CONFIGURATION ---
NEWS_FILE_PATH = '../data/raw_analyst_ratings.csv' 
TICKERS_TO_ANALYZE = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA'] 
HEADLINE_COLUMN = 'headline'
DATE_COLUMN = 'date'
STOCK_COLUMN = 'stock' 

# --- STEP 2: INITIALIZE ANALYZER CLASS ---
analyzer = FinancialAnalyzer()

# --- STEP 3: LOAD AND PRE-PROCESS NEWS DATA ---
print(f"\nLoading and processing data from {NEWS_FILE_PATH}...")
try:
    news_df = pd.read_csv(NEWS_FILE_PATH)
    
    # 1. Prepare data 
    news_df.rename(columns={DATE_COLUMN: 'Date', STOCK_COLUMN: 'stock'}, inplace=True)
    news_df['Date'] = pd.to_datetime(news_df['Date'], errors='coerce').dt.date
    news_df.dropna(subset=['Date', 'stock'], inplace=True)
    
    # 2. Calculate Sentiment using the class method
    print("Calculating VADER sentiment scores for all headlines...")
    # Use the class method with 'apply'
    news_df['Sentiment_Score'] = news_df[HEADLINE_COLUMN].apply(analyzer.calculate_vader_score)

    # 3. AGGREGATION using the class method
    df_news_agg = analyzer.aggregate_sentiment_by_day_and_stock(
        df=news_df, 
        date_col='Date', 
        stock_col='stock', 
        sentiment_col='Sentiment_Score'
    )
    
    print(f"\nAggregation complete. Generated {len(df_news_agg)} daily sentiment records.")

except FileNotFoundError:
    print(f"ERROR: The news file '{NEWS_FILE_PATH}' was not found. Please check the file path.")
    df_news_agg = pd.DataFrame() # Create empty DataFrame to prevent errors
except Exception as e:
    print(f"An unexpected error occurred during news processing: {e}")
    df_news_agg = pd.DataFrame()

# --- STEP 4: RUN CORRELATION ANALYSIS (END-TO-END) ---

if not df_news_agg.empty:
    print("\n--- Starting End-to-End Correlation Analysis ---")

    results = {}

    for TICKER in TICKERS_TO_ANALYZE:
        STOCK_FILE_PATH = f'{TICKER}.csv' 
        
        # Call the class method to handle loading, merging, and correlation
        correlation = analyzer.correlate_sentiment_with_returns(
            stock_ticker=TICKER,
            stock_file_path=STOCK_FILE_PATH,
            df_news_agg=df_news_agg
        )
        if correlation is not None:
            results[TICKER] = correlation

    # Display Final Summary
    print("\n==============================================")
    print("FINAL CORRELATION SUMMARY (Sentiment vs. Daily Return)")
    print("==============================================")
    for ticker, corr in results.items():
        print(f"| {ticker:<4} | Correlation: {corr:.4f} |")
    print("==============================================")

else:
    print("Cannot run correlation. News data aggregation failed.")

System path updated for module import.
--- Initializing FinancialAnalyzer and VADER Sentiment Analyzer ---
VADER Sentiment Analyzer initialized and ready.

Loading and processing data from ../data/raw_analyst_ratings.csv...
Calculating VADER sentiment scores for all headlines...

Aggregation complete. Generated 44196 daily sentiment records.

--- Starting End-to-End Correlation Analysis ---
[AAPL] An unexpected error occurred during correlation analysis: isinstance() arg 2 must be a type, a tuple of types, or a union
[AMZN] An unexpected error occurred during correlation analysis: isinstance() arg 2 must be a type, a tuple of types, or a union
[GOOG] An unexpected error occurred during correlation analysis: isinstance() arg 2 must be a type, a tuple of types, or a union
[META] WARNING: Insufficient matching data points for correlation. (Points: 0)
[MSFT] WARNING: Insufficient matching data points for correlation. (Points: 0)
[NVDA] An unexpected error occurred during correlation analys